In [2]:
# Cell 1 — Ultimate Offline vLLM Installer (Enforcing Python 3.12)
import subprocess
import sys
from pathlib import Path

try:
    import vllm  # noqa: F401
    print('vLLM already installed, skipping wheel install.')
except ImportError:
    print("Scanning /kaggle/input/ for offline wheels...")
    
    # --- STEP 1: FORCE INSTALL THE PYTHON 3.12 NUMPY PATCH ---
    # We strictly search for "cp312" so it ignores older cp310 wheels lurking in other datasets
    numpy_patch_files = list(Path('/kaggle/input').glob('**/numpy-1.26*cp312*.whl'))
    
    if numpy_patch_files:
        numpy_wheel = numpy_patch_files[0]
        print(f"Installing Numpy Patch directly: {numpy_wheel.name}")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', str(numpy_wheel), '--no-index'])
    else:
        print("Warning: Numpy 1.26 cp312 patch not found! Did you attach the dataset?")

    # --- STEP 2: INSTALL vLLM ---
    # Find every directory in Kaggle inputs that contains a .whl file
    wheel_dirs = list({p.parent for p in Path('/kaggle/input').glob('**/*.whl')})
    
    if not wheel_dirs:
        raise RuntimeError("No wheels found! Please attach a vLLM wheel dataset.")
    
    args = [sys.executable, '-m', 'pip', 'install', 'vllm', '--no-index']
    for d in wheel_dirs:
        args.extend(['--find-links', str(d)])
        
    print(f"\nInstalling vLLM using offline directories: {[d.name for d in wheel_dirs]}")
    
    try:
        subprocess.check_call(args)
        print('\nvLLM installed successfully.')
    except subprocess.CalledProcessError as e:
        print(f"\nCRITICAL ERROR: Pip failed. Missing a dependency wheel in your attached datasets.")
        raise e

print('vLLM ready')

Scanning /kaggle/input/ for offline wheels...
Installing Numpy Patch directly: numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/datasets/aurascoper/numpy-1-26-4-cp312/numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 


Installing vLLM using offline directories: ['numpy-1-26-4-cp312', 'vllm-0-7-1', 'arc_agi_3_wheels']
Looking in links: /kaggle/input/datasets/aurascoper/numpy-1-26-4-cp312, /kaggle/input/datasets/konstantinboyko/vllm-0-7-1, /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/datasets/konstantinboyko/vllm-0-7-1/vllm-0.7.1-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/input/datasets/konstantinboyko/vllm-0-7-1/prometheus_fastapi_instrumentator-7.0.2-py3-none-any.whl (from vllm)
Processing /kaggle/input/datasets/konstantinboyko/vllm-0-7-1/lm_format_enforcer-0.10.9-py3-none-any.whl (from vllm)
Processing /kaggle/input/datasets/konstantinboyko/vllm-0-7-1/outlines-0.1.11-py3-none-any.whl (from vllm)
Processing /kaggle/input/datasets/konstantinboyko/vllm-0-7-1/lark-1.2.2-py3-none-any.whl (from vllm)
INFO: pip is looking at multiple versions of vllm to determine which version is compatible with other requirements. This could take a while.

CRITICAL

ERROR: Could not find a version that satisfies the requirement xgrammar>=0.1.6; platform_machine == "x86_64" (from vllm) (from versions: none)
ERROR: No matching distribution found for xgrammar>=0.1.6; platform_machine == "x86_64"


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', 'vllm', '--no-index', '--find-links', '/kaggle/input/datasets/aurascoper/numpy-1-26-4-cp312', '--find-links', '/kaggle/input/datasets/konstantinboyko/vllm-0-7-1', '--find-links', '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels']' returned non-zero exit status 1.

In [ ]:
# Cell 2: Environment Setup — Kaggle T4×2
import os

os.environ["ARC_MODEL_PATH"] = "/kaggle/input/models/qwen-lm/qwen-3/transformers/30b-a3b-thinking-2507-fp8/1"
os.environ["ARC_TP"] = "2"
os.environ["ARC_QUANTIZATION"] = "fp8"
# 3072: thinking block ~500-1500 tokens + code answer ~300-500 tokens
# 1024 cuts off mid-reasoning; 4096+ risks timeout on 240 tasks
os.environ["ARC_MAX_TOKENS"] = "3072"

print(f"Model: {os.environ['ARC_MODEL_PATH']}")
print(f"TP:    {os.environ['ARC_TP']}")
print(f"Max tokens: {os.environ['ARC_MAX_TOKENS']}")

In [ ]:
# Cell 3 — Copy dsl.py and solver from aurascoper/modules dataset
import shutil
from pathlib import Path

WORKING = Path('/kaggle/working')
MODULES = Path('/kaggle/input/modules')

shutil.copy(MODULES / 'dsl.py', WORKING / 'dsl.py')
print(f'Copied dsl.py from {MODULES / "dsl.py"}')

shutil.copy(MODULES / 'target_arc2_kaggle.py', WORKING / 'target_arc2_kaggle.py')
print(f'Copied solver from {MODULES / "target_arc2_kaggle.py"}')